In [2]:
# =====================================================================
# 10 - TÜRKİYE SAĞLIK BÜTÇESİ TAHMİNİ
# STABİL SÜRÜM
#
# Temel değişiklik:
#   1) TL bütçe doğrudan tahmin edilmiyor.
#   2) Model hedefi: butce_USD, yani USD bazlı toplam sağlık harcaması.
#   3) Recursive lag tahmini kaldırıldı.
#   4) 2025-2030 USD/TRY kuru modelle tahmin edilmiyor.
#      Direkt usdtry_aylik_tahmin_2026_2030.csv dosyasından alınıyor.
#   5) Nihai TL bütçe şu şekilde türetiliyor:
#      butce_TL_turetilen = butce_USD_tahmin × usdtry_hazir_veri
#
# Amaç:
#   Recursive prediction kaynaklı exponential şişmeyi engellemek.
# =====================================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet, HuberRegressor, TheilSenRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


In [3]:
# %% [2] BÜTÇE VERİSİ OKUMA ==========================================

BUTCE_KAYNAKLAR = [
    "99-24saglikbutcesi_temiz.xlsx",
    "99-24saglikbutcesi_temix.xlsx",  # olası yazım farkı için
    "99-24saglikbutcesi_temiz.csv",
    "99-24saglikbutcesi.xlsx",
    "99-24saglikbutcesi.csv",
]

butce_path = next((p for p in BUTCE_KAYNAKLAR if os.path.exists(p)), None)
if butce_path is None:
    raise FileNotFoundError("Bütçe dosyası bulunamadı. Beklenen dosya: 99-24saglikbutcesi_temiz.xlsx")

# Bu dosyada gerçek kolon başlıkları 3. satırda olduğu için header=2 kullanılıyor.
if butce_path.endswith((".xls", ".xlsx")):
    raw = pd.read_excel(butce_path, header=2)
else:
    raw = pd.read_csv(butce_path, header=2)

raw.columns = [str(c).strip() for c in raw.columns]
print(f"[OK] Bütçe dosyası okundu: {butce_path}")
print("Sütunlar:", raw.columns.tolist())

butce = raw.rename(columns={
    "Yıl": "yıl",
    "Toplam sağlık harcaması (Milyon TL)": "butce_TL",
    "Kişi başına sağlık harcaması (TL)": "kisi_basi_TL",
    "Genel devlet sağlık harcaması oranı (%)": "devlet_orani",
    "Özel sektör sağlık harcaması oranı (%)": "ozel_orani",
}).copy()

gerekli = ["yıl", "butce_TL", "kisi_basi_TL", "devlet_orani", "ozel_orani"]
eksik = [c for c in gerekli if c not in butce.columns]
if eksik:
    raise KeyError(f"Bütçe dosyasında beklenen sütunlar bulunamadı: {eksik}")

butce = butce[gerekli]

for col in gerekli:
    butce[col] = pd.to_numeric(
        butce[col].astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace(" ", "", regex=False),
        errors="coerce"
    )

butce = butce.dropna(subset=["yıl", "butce_TL"]).copy()
butce["yıl"] = butce["yıl"].astype(int)
butce = butce[(butce["yıl"] >= 1999) & (butce["yıl"] <= 2024)].sort_values("yıl").reset_index(drop=True)

print("\nBütçe verisi:")
display(butce)


[OK] Bütçe dosyası okundu: 99-24saglikbutcesi_temiz.xlsx
Sütunlar: ['Yıl', 'Toplam sağlık harcaması (Milyon TL)', 'Kişi başına sağlık harcaması (TL)', 'Genel devlet sağlık harcaması oranı (%)', 'Özel sektör sağlık harcaması oranı (%)']

Bütçe verisi:


,yıl,butce_TL,kisi_basi_TL,devlet_orani,ozel_orani
0,1999,4.985000e+03,79.00000,61.1,38.9
1,2000,8.248000e+03,128.00000,62.9,37.1
2,2001,1.239588e+04,190.00000,68.1,31.9
3,2002,1.877394e+04,284.00000,70.7,29.3
4,2003,2.427891e+04,363.00000,71.9,28.1
5,2004,3.002085e+04,444.00000,71.2,28.8
6,2005,3.535891e+04,517.00000,67.8,32.2
7,2006,4.406868e+04,636.00000,68.3,31.7
8,2007,5.090430e+04,726.00000,67.8,32.2
9,2008,5.773961e+04,813.00000,73.0,27.0


In [4]:
# %% [3] TARİHSEL USD/TRY KURU — YILLIK ORTALAMA =====================

KUR_PATHS = ["usd-try.csv", "usd-try-8a0a40c9.csv"]
kur_path = next((p for p in KUR_PATHS if os.path.exists(p)), None)
if kur_path is None:
    raise FileNotFoundError("usd-try.csv bulunamadı.")

kur = pd.read_csv(kur_path)
kur.columns = [str(c).strip() for c in kur.columns]

if "Tarih" not in kur.columns or "Şimdi" not in kur.columns:
    raise KeyError("usd-try.csv içinde 'Tarih' ve 'Şimdi' sütunları bekleniyor.")

kur["tarih"] = pd.to_datetime(kur["Tarih"], format="%d.%m.%Y", errors="coerce")
kur["kur"] = pd.to_numeric(
    kur["Şimdi"].astype(str)
    .str.replace(",", ".", regex=False)
    .str.replace(" ", "", regex=False),
    errors="coerce"
)
kur["yıl"] = kur["tarih"].dt.year

kur_yillik = (
    kur.dropna(subset=["yıl", "kur"])
    .groupby("yıl", as_index=False)["kur"]
    .mean()
    .rename(columns={"kur": "usdtry"})
)
kur_yillik["yıl"] = kur_yillik["yıl"].astype(int)

print(f"[OK] Tarihsel kur okundu: {kur_path}")
print(f"Yıl aralığı: {kur_yillik['yıl'].min()}-{kur_yillik['yıl'].max()}")
display(kur_yillik.tail())


[OK] Tarihsel kur okundu: usd-try.csv
Yıl aralığı: 1999-2024


,yıl,usdtry
21,2020,7.093308
22,2021,9.204225
23,2022,16.683283
24,2023,24.221317
25,2024,33.065233


In [5]:
# %% [4] 2025-2030 HAZIR USD/TRY KUR VERİSİ ==========================
# Burada USD/TRY tahmini yapılmaz. Dosyadaki aylık tahminler yıllık ortalamaya çevrilir.

GELECEK_KUR_PATHS = [
    "usdtry_aylik_tahmin_2026_2030.csv",
    "usdtry_aylik_tahmin_2025_2030.csv",
]

gelecek_kur_path = next((p for p in GELECEK_KUR_PATHS if os.path.exists(p)), None)
if gelecek_kur_path is None:
    raise FileNotFoundError("Gelecek USD/TRY dosyası bulunamadı. Beklenen: usdtry_aylik_tahmin_2026_2030.csv")

gelecek_kur = pd.read_csv(gelecek_kur_path)
gelecek_kur.columns = [str(c).strip().lower() for c in gelecek_kur.columns]

# Yıl kolonu bulma
yil_col = None
for aday in ["yıl", "yil", "year"]:
    if aday in gelecek_kur.columns:
        yil_col = aday
        break

if yil_col is None:
    tarih_adaylari = [c for c in gelecek_kur.columns if "tarih" in c or "date" in c]
    if not tarih_adaylari:
        raise KeyError("Gelecek kur dosyasında yıl/yil/year veya tarih/date sütunu bulunamadı.")
    tarih_col = tarih_adaylari[0]
    gelecek_kur[tarih_col] = pd.to_datetime(gelecek_kur[tarih_col], errors="coerce")
    gelecek_kur["yıl"] = gelecek_kur[tarih_col].dt.year
    yil_col = "yıl"

# Kur kolonu bulma
kur_col = None
for aday in ["usdtry", "usd_try", "usd/try", "kur", "tahmin", "forecast", "prediction"]:
    if aday in gelecek_kur.columns:
        kur_col = aday
        break

if kur_col is None:
    adaylar = [c for c in gelecek_kur.columns if "usd" in c or "kur" in c or "try" in c]
    if not adaylar:
        raise KeyError("Gelecek kur dosyasında kur/usdtry sütunu bulunamadı.")
    kur_col = adaylar[0]

gelecek_kur[yil_col] = pd.to_numeric(gelecek_kur[yil_col], errors="coerce")
gelecek_kur[kur_col] = pd.to_numeric(
    gelecek_kur[kur_col].astype(str)
    .str.replace(",", ".", regex=False)
    .str.replace(" ", "", regex=False),
    errors="coerce"
)

GELECEK_YILLAR = list(range(2025, 2031))

gelecek_kur_yillik = (
    gelecek_kur.dropna(subset=[yil_col, kur_col])
    .groupby(yil_col, as_index=False)[kur_col]
    .mean()
    .rename(columns={yil_col: "yıl", kur_col: "usdtry_hazir_veri"})
)
gelecek_kur_yillik["yıl"] = gelecek_kur_yillik["yıl"].astype(int)
gelecek_kur_yillik = gelecek_kur_yillik[gelecek_kur_yillik["yıl"].isin(GELECEK_YILLAR)].sort_values("yıl").reset_index(drop=True)

eksik_yillar = sorted(set(GELECEK_YILLAR) - set(gelecek_kur_yillik["yıl"]))
if eksik_yillar:
    raise ValueError(f"Gelecek kur dosyasında eksik yıllar var: {eksik_yillar}")

print(f"[OK] Gelecek kur dosyası okundu: {gelecek_kur_path}")
print("2025-2030 yıllık ortalama USD/TRY:")
display(gelecek_kur_yillik)


[OK] Gelecek kur dosyası okundu: usdtry_aylik_tahmin_2026_2030.csv
2025-2030 yıllık ortalama USD/TRY:


,yıl,usdtry_hazir_veri
0,2025,39.542500
1,2026,46.440833
2,2027,55.930000
3,2028,66.080000
4,2029,83.172500
5,2030,96.180000


In [6]:
# %% [5] BÜTÇE + KUR BİRLEŞTİRME ====================================

veri = butce.merge(kur_yillik, on="yıl", how="left")

kur_eksik = veri.loc[veri["usdtry"].isna(), "yıl"].tolist()
if kur_eksik:
    raise ValueError(f"Tarihsel USD/TRY kuru eksik yıllar: {kur_eksik}")

# Ana hedef: USD bazlı toplam sağlık harcaması
veri["butce_USD"] = veri["butce_TL"] / veri["usdtry"]
veri["kisi_basi_USD"] = veri["kisi_basi_TL"] / veri["usdtry"]
veri["butce_USD_yoy_%"] = veri["butce_USD"].pct_change() * 100

veri = veri.sort_values("yıl").reset_index(drop=True)

print("\nTarihsel veri — son 10 yıl:")
display(veri[["yıl", "butce_TL", "usdtry", "butce_USD", "butce_USD_yoy_%"]].tail(10))

veri.to_csv("butce_veri_hazir_stabil.csv", index=False, encoding="utf-8-sig")



Tarihsel veri — son 10 yıl:


,yıl,butce_TL,usdtry,butce_USD,butce_USD_yoy_%
16,2015,1.045675e+05,2.751158,38008.550059,-12.103852
17,2016,1.197558e+05,3.029242,39533.253901,4.011476
18,2017,1.406473e+05,3.639775,38641.770112,-2.255023
19,2018,1.652340e+05,4.853192,34046.468935,-11.892057
20,2019,2.010306e+05,5.673725,35431.853353,4.069099
21,2020,2.499319e+05,7.093308,35234.878693,-0.555925
22,2021,3.539410e+05,9.204225,38454.184527,9.136702
23,2022,6.068352e+05,16.683283,36373.848962,-5.409907
24,2023,1.244237e+06,24.221317,51369.515963,41.226506
25,2024,2.359151e+06,33.065233,71348.393413,38.892478


In [7]:
# %% [6] MODEL AYARLARI ==============================================
# Not:
# - Recursive lag kullanılmıyor.
# - Modeller doğrudan 2025-2030 ufkunu tahmin ediyor.
# - Hedef log(USD bütçe) olan modeller pozitif ve daha stabil sonuç verir.
# - Tree modelleri de denenir; fakat nihai ensemble seçimi validation performansına göre yapılır.

HEDEF = "butce_USD"
YIL_COL = "yıl"

# Makro gerçeklik kontrolü:
# Sağlık bütçesi USD bazında yıldan yıla çok sert zıplamamalı.
# Bu guardrail recursive explosion veya aşırı trend patlamasını engeller.
# İstersen daha esnek yapmak için MAX_USD_YOY değerini 0.15 yapabilirsin.
MIN_USD_YOY = -0.05   # -%5 altına düşüşü sınırlama
MAX_USD_YOY =  0.12   # +%12 üstüne artışı sınırlama

TOP_K_ENSEMBLE = 5

MODEL_ADLARI = [
    "LogLinear",
    "LogRidge",
    "LogElasticNet",
    "LogPoly2Ridge",
    "LogHuber",
    "TheilSen",
    "RandomForest",
    "ExtraTrees",
    "GradientBoosting",
    "HoltDampedAdd",
    "HoltAdd",
    "ARIMA",
    "CAGR5",
    "CAGR10",
    "CAGR15",
]

print(f"Hedef değişken: {HEDEF}")
print(f"Nihai ensemble model sayısı: {TOP_K_ENSEMBLE}")
print(f"USD yıllık büyüme guardrail: {MIN_USD_YOY*100:.0f}% ile {MAX_USD_YOY*100:.0f}% arası")


Hedef değişken: butce_USD
Nihai ensemble model sayısı: 5
USD yıllık büyüme guardrail: -5% ile 12% arası


In [8]:
# %% [7] MODEL FONKSİYONLARI =========================================

def make_X(years, origin=1999):
    """Yılı modele uygun tek değişkenli feature'a çevirir."""
    return (np.asarray(years, dtype=float) - origin).reshape(-1, 1)


def apply_growth_guardrail(pred, base_value, min_growth=MIN_USD_YOY, max_growth=MAX_USD_YOY):
    """Tahminleri yıldan yıla makul büyüme bandına alır."""
    pred = np.asarray(pred, dtype=float)
    fixed = []
    prev = float(base_value)
    for val in pred:
        lower = prev * (1 + min_growth)
        upper = prev * (1 + max_growth)
        new_val = float(np.clip(val, lower, upper))
        fixed.append(new_val)
        prev = new_val
    return np.asarray(fixed, dtype=float)


def forecast_model(model_name, train_years, train_y, future_years, use_guardrail=True):
    """Verilen model ile doğrudan gelecek ufku tahmini yapar."""
    train_years = np.asarray(train_years, dtype=int)
    train_y = np.asarray(train_y, dtype=float)
    future_years = np.asarray(future_years, dtype=int)

    X = make_X(train_years)
    Xf = make_X(future_years)
    h = len(future_years)

    if model_name == "LogLinear":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ])
        model.fit(X, np.log(train_y))
        pred = np.exp(model.predict(Xf))

    elif model_name == "LogRidge":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10.0, random_state=RANDOM_STATE))
        ])
        model.fit(X, np.log(train_y))
        pred = np.exp(model.predict(Xf))

    elif model_name == "LogElasticNet":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.02, l1_ratio=0.20, max_iter=20000, random_state=RANDOM_STATE))
        ])
        model.fit(X, np.log(train_y))
        pred = np.exp(model.predict(Xf))

    elif model_name == "LogPoly2Ridge":
        model = Pipeline([
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=30.0, random_state=RANDOM_STATE))
        ])
        model.fit(X, np.log(train_y))
        pred = np.exp(model.predict(Xf))

    elif model_name == "LogHuber":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", HuberRegressor(epsilon=1.5, alpha=1.0, max_iter=2000))
        ])
        model.fit(X, np.log(train_y))
        pred = np.exp(model.predict(Xf))

    elif model_name == "TheilSen":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", TheilSenRegressor(random_state=RANDOM_STATE))
        ])
        model.fit(X, train_y)
        pred = model.predict(Xf)

    elif model_name == "RandomForest":
        model = RandomForestRegressor(
            n_estimators=500,
            max_depth=3,
            min_samples_leaf=3,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        model.fit(X, train_y)
        pred = model.predict(Xf)

    elif model_name == "ExtraTrees":
        model = ExtraTreesRegressor(
            n_estimators=500,
            max_depth=3,
            min_samples_leaf=3,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        model.fit(X, train_y)
        pred = model.predict(Xf)

    elif model_name == "GradientBoosting":
        model = GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=3,
            random_state=RANDOM_STATE
        )
        model.fit(X, train_y)
        pred = model.predict(Xf)

    elif model_name == "HoltDampedAdd":
        model = ExponentialSmoothing(
            train_y,
            trend="add",
            damped_trend=True,
            seasonal=None,
            initialization_method="estimated"
        ).fit(optimized=True)
        pred = model.forecast(h)

    elif model_name == "HoltAdd":
        model = ExponentialSmoothing(
            train_y,
            trend="add",
            damped_trend=False,
            seasonal=None,
            initialization_method="estimated"
        ).fit(optimized=True)
        pred = model.forecast(h)

    elif model_name == "ARIMA":
        best_model = None
        best_aic = np.inf
        for order in [(0, 1, 1), (1, 1, 0), (1, 1, 1), (2, 1, 0), (0, 1, 2)]:
            try:
                candidate = ARIMA(train_y, order=order).fit()
                if candidate.aic < best_aic:
                    best_model = candidate
                    best_aic = candidate.aic
            except Exception:
                continue
        if best_model is None:
            pred = np.repeat(train_y[-1], h)
        else:
            pred = best_model.forecast(h)

    elif model_name in ["CAGR5", "CAGR10", "CAGR15"]:
        n = int(model_name.replace("CAGR", ""))
        n = min(n, len(train_y))
        if n > 1 and train_y[-n] > 0:
            g = (train_y[-1] / train_y[-n]) ** (1 / (n - 1)) - 1
        else:
            g = 0.0
        pred = np.array([train_y[-1] * (1 + g) ** step for step in range(1, h + 1)])

    else:
        raise ValueError(f"Bilinmeyen model: {model_name}")

    pred = np.clip(np.asarray(pred, dtype=float), 1e-6, None)

    if use_guardrail:
        pred = apply_growth_guardrail(pred, base_value=train_y[-1])

    return pred


def metrikler(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    accuracy = max(0, 100 - mape)

    return {
        "Accuracy": accuracy,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "ErrorRate": mape,
    }


In [9]:
# %% [8] ROLLING VALIDATION ==========================================
# Veri az olduğu için klasik random train-test yerine zamana duyarlı rolling validation kullanılır.

folds = [
    (range(1999, 2017), range(2017, 2020)),
    (range(1999, 2018), range(2018, 2021)),
    (range(1999, 2019), range(2019, 2022)),
    (range(1999, 2020), range(2020, 2023)),
    (range(1999, 2022), range(2022, 2025)),  # 2022-2024 son test bloğu
]

years_all = veri[YIL_COL].values.astype(int)
y_all = veri[HEDEF].values.astype(float)

sonuclar = []
fold5_tahminler = []

for model_name in MODEL_ADLARI:
    for fold_id, (train_years, test_years) in enumerate(folds, 1):
        train_years = list(train_years)
        test_years = list(test_years)

        train_mask = veri[YIL_COL].isin(train_years).values
        test_mask = veri[YIL_COL].isin(test_years).values

        if train_mask.sum() < 8 or test_mask.sum() == 0:
            continue

        tr_years = years_all[train_mask]
        tr_y = y_all[train_mask]
        te_years = years_all[test_mask]
        te_y = y_all[test_mask]

        try:
            pred = forecast_model(model_name, tr_years, tr_y, te_years, use_guardrail=True)
        except Exception as e:
            print(f"[HATA] {model_name} Fold {fold_id}: {e}")
            continue

        met = metrikler(te_y, pred)
        met.update({
            "Model": model_name,
            "Fold": fold_id,
            "TrainSon": int(max(tr_years)),
            "TestBas": int(min(te_years)),
            "TestSon": int(max(te_years)),
        })
        sonuclar.append(met)

        if fold_id == 5:
            for yr, gercek, tah in zip(te_years, te_y, pred):
                fold5_tahminler.append({
                    "yıl": int(yr),
                    "Model": model_name,
                    "gercek_butce_USD": float(gercek),
                    "tahmin_butce_USD": float(tah),
                    "hata": float(gercek - tah),
                })

    print(f"{model_name:<18} bitti")

sdf = pd.DataFrame(sonuclar)
fold5_df = pd.DataFrame(fold5_tahminler)

sdf.to_csv("butce_metrikleri_detay_stabil.csv", index=False, encoding="utf-8-sig")
fold5_df.to_csv("butce_fold5_tahminleri_stabil.csv", index=False, encoding="utf-8-sig")

print("\n[OK] Rolling validation tamamlandı.")


LogLinear          bitti
LogRidge           bitti
LogElasticNet      bitti
LogPoly2Ridge      bitti
LogHuber           bitti
TheilSen           bitti
RandomForest       bitti
ExtraTrees         bitti
GradientBoosting   bitti
HoltDampedAdd      bitti


C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
C:\Users\Bedirhan\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimiz

HoltAdd            bitti
ARIMA              bitti
CAGR5              bitti
CAGR10             bitti
CAGR15             bitti

[OK] Rolling validation tamamlandı.


In [10]:
# %% [9] MODEL KARŞILAŞTIRMA =========================================

genel = (
    sdf.groupby("Model")[["Accuracy", "MAE", "RMSE", "R2", "ErrorRate"]]
    .mean()
    .round(3)
    .reset_index()
    .sort_values("MAE")
)

# Ağırlıklı ensemble için MAE düşük olan modeller tercih edilir.
# Aşırı overfit tek model yerine top-k ensemble daha stabil sonuç verir.
top_modeller = genel.head(TOP_K_ENSEMBLE).copy()

top_modeller["weight_raw"] = 1 / (top_modeller["MAE"] + 1e-6)
top_modeller["ensemble_weight"] = top_modeller["weight_raw"] / top_modeller["weight_raw"].sum()
top_modeller = top_modeller.drop(columns=["weight_raw"])

genel.to_csv("butce_metrikleri_genel_stabil.csv", index=False, encoding="utf-8-sig")
top_modeller.to_csv("butce_ensemble_modelleri_stabil.csv", index=False, encoding="utf-8-sig")

print("========== GENEL MODEL KARŞILAŞTIRMASI — USD HEDEF, STABİL ==========")
display(genel)

print("\n========== ENSEMBLE İÇİN SEÇİLEN MODELLER ==========")
display(top_modeller)


========== GENEL MODEL KARŞILAŞTIRMASI — USD HEDEF, STABİL ==========


,Model,Accuracy,MAE,RMSE,R2,ErrorRate
4,ExtraTrees,90.210,4965.371,6157.454,-4.891,9.790
0,ARIMA,88.114,5558.844,6618.294,-8.782,11.886
1,CAGR10,88.370,5570.594,6826.324,-6.496,11.630
3,CAGR5,88.153,5633.133,6962.098,-6.648,11.847
5,GradientBoosting,88.254,5661.530,6883.718,-9.977,11.746
6,HoltAdd,86.677,6011.173,7147.547,-8.919,13.323
13,RandomForest,86.786,6126.670,7348.851,-17.982,13.214
7,HoltDampedAdd,86.119,6255.726,7406.740,-16.535,13.881
2,CAGR15,82.573,7537.832,8698.352,-50.974,17.427
11,LogPoly2Ridge,74.757,9961.716,10909.559,-100.430,25.243



========== ENSEMBLE İÇİN SEÇİLEN MODELLER ==========


,Model,Accuracy,MAE,RMSE,R2,ErrorRate,ensemble_weight
4,ExtraTrees,90.210,4965.371,6157.454,-4.891,9.790,0.220115
0,ARIMA,88.114,5558.844,6618.294,-8.782,11.886,0.196615
1,CAGR10,88.370,5570.594,6826.324,-6.496,11.630,0.196200
3,CAGR5,88.153,5633.133,6962.098,-6.648,11.847,0.194022
5,GradientBoosting,88.254,5661.530,6883.718,-9.977,11.746,0.193049


In [11]:
# %% [10] 2025-2030 USD BÜTÇE TAHMİNİ — DIRECT FORECAST ==============
# Burada recursive lag yoktur.
# Her model 2025-2030 ufkunu doğrudan tahmin eder.

future_years = np.array(GELECEK_YILLAR, dtype=int)
model_forecasts = []

for _, row in top_modeller.iterrows():
    model_name = row["Model"]
    weight = row["ensemble_weight"]

    pred = forecast_model(model_name, years_all, y_all, future_years, use_guardrail=True)

    temp = pd.DataFrame({
        "yıl": future_years,
        "Model": model_name,
        "weight": weight,
        "butce_USD_tahmin": pred,
    })
    model_forecasts.append(temp)

model_forecasts_df = pd.concat(model_forecasts, ignore_index=True)
model_forecasts_df.to_csv("butce_model_bazli_2025_2030_stabil.csv", index=False, encoding="utf-8-sig")

# Ağırlıklı ensemble
pred_matrix = []
weights = []
for _, row in top_modeller.iterrows():
    model_name = row["Model"]
    weights.append(row["ensemble_weight"])
    pred_matrix.append(
        model_forecasts_df.loc[model_forecasts_df["Model"] == model_name, "butce_USD_tahmin"].values
    )

pred_matrix = np.vstack(pred_matrix)
weights = np.asarray(weights, dtype=float)

pred_usd_ensemble = np.average(pred_matrix, axis=0, weights=weights)

# Ensemble sonrasında bir kez daha büyüme bandı kontrolü
pred_usd_ensemble = apply_growth_guardrail(pred_usd_ensemble, base_value=y_all[-1])

print("Model bazlı 2025-2030 USD tahminleri:")
display(model_forecasts_df.pivot_table(index="yıl", columns="Model", values="butce_USD_tahmin").round(2))


Model bazlı 2025-2030 USD tahminleri:


Model,ARIMA,CAGR10,CAGR5,ExtraTrees,GradientBoosting
yıl,,,,,
2025,79910.20,76519.73,79910.20,67780.97,67780.97
2026,87155.38,82065.88,89499.42,64391.93,64391.93
2027,89970.81,88014.01,100239.36,61172.33,61172.33
2028,91435.92,94393.27,112268.08,58113.71,58113.71
2029,92198.34,101234.90,125740.25,55208.03,55208.03
2030,92595.09,108572.41,140829.08,52447.63,52447.63


In [12]:
# %% [11] USD'DEN TL'YE ÇEVİRME VE ÇIKTI =============================

kur_dict = dict(zip(gelecek_kur_yillik["yıl"], gelecek_kur_yillik["usdtry_hazir_veri"]))
usdtry_future = np.array([kur_dict[y] for y in future_years], dtype=float)

pred_tl_ensemble = pred_usd_ensemble * usdtry_future

sonuc = pd.DataFrame({
    "yıl": future_years,
    "butce_USD_tahmin (Milyon USD)": np.round(pred_usd_ensemble, 2),
    "usdtry_hazir_veri": np.round(usdtry_future, 4),
    "butce_TL_turetilen (Milyon TL)": np.round(pred_tl_ensemble, 2),
})

sonuc["usd_yoy_%"] = sonuc["butce_USD_tahmin (Milyon USD)"].pct_change() * 100
sonuc["tl_yoy_%"] = sonuc["butce_TL_turetilen (Milyon TL)"].pct_change() * 100

print("========== 2025-2030 BÜTÇE TAHMİNİ — STABİL ENSEMBLE ==========")
display(sonuc)

print("\nYıllık büyüme kontrolü:")
display(sonuc[["yıl", "usd_yoy_%", "tl_yoy_%"]].round(2))

# Tarihsel + tahmin birleşik veri
tarihsel = veri[["yıl", "butce_TL", "usdtry", "butce_USD"]].copy()
tarihsel.columns = ["yıl", "butce_TL", "usdtry", "butce_USD"]
tarihsel["tip"] = "Gerçek"

tahmin = pd.DataFrame({
    "yıl": future_years,
    "butce_TL": pred_tl_ensemble,
    "usdtry": usdtry_future,
    "butce_USD": pred_usd_ensemble,
    "tip": "Tahmin"
})

birlesik = pd.concat([tarihsel, tahmin], ignore_index=True)

sonuc.to_csv("butce_2025_2030_tahmin_stabil.csv", index=False, encoding="utf-8-sig")
sonuc.to_excel("butce_2025_2030_tahmin_stabil.xlsx", index=False)

birlesik.to_csv("butce_tarihsel_tahmin_birlesik_stabil.csv", index=False, encoding="utf-8-sig")
birlesik.to_excel("butce_tarihsel_tahmin_birlesik_stabil.xlsx", index=False)

print("\n[OK] Çıktılar oluşturuldu:")
print("  - butce_2025_2030_tahmin_stabil.xlsx")
print("  - butce_2025_2030_tahmin_stabil.csv")
print("  - butce_tarihsel_tahmin_birlesik_stabil.xlsx")
print("  - butce_tarihsel_tahmin_birlesik_stabil.csv")
print("  - butce_metrikleri_genel_stabil.csv")
print("  - butce_ensemble_modelleri_stabil.csv")
print("  - butce_model_bazli_2025_2030_stabil.csv")


========== 2025-2030 BÜTÇE TAHMİNİ — STABİL ENSEMBLE ==========


,yıl,butce_USD_tahmin (Milyon USD),usdtry_hazir_veri,butce_TL_turetilen (Milyon TL),usd_yoy_%,tl_yoy_%
0,2025,74233.64,39.5425,2935383.62,NaN,NaN
1,2026,77206.59,46.4408,3585538.43,4.004855,22.148887
2,2027,79680.73,55.9300,4456543.41,3.204571,24.292167
3,2028,82290.53,66.0800,5437758.39,3.275321,22.017400
4,2029,85196.14,83.1725,7085975.58,3.530917,30.310600
5,2030,88500.83,96.1800,8512009.65,3.878920,20.124739



Yıllık büyüme kontrolü:


,yıl,usd_yoy_%,tl_yoy_%
0,2025,NaN,NaN
1,2026,4.00,22.15
2,2027,3.20,24.29
3,2028,3.28,22.02
4,2029,3.53,30.31
5,2030,3.88,20.12



[OK] Çıktılar oluşturuldu:
  - butce_2025_2030_tahmin_stabil.xlsx
  - butce_2025_2030_tahmin_stabil.csv
  - butce_tarihsel_tahmin_birlesik_stabil.xlsx
  - butce_tarihsel_tahmin_birlesik_stabil.csv
  - butce_metrikleri_genel_stabil.csv
  - butce_ensemble_modelleri_stabil.csv
  - butce_model_bazli_2025_2030_stabil.csv


In [13]:
# %% [12] KISA YORUM ================================================

print("""
Yorum:
- Bu sürümde 2025-2030 için USD/TRY tahmini yapılmaz; hazır kur dosyasından alınır.
- Sağlık bütçesi önce USD bazında tahmin edilir.
- TL bütçe daha sonra USD tahmini × hazır USD/TRY kuru ile türetilir.
- Recursive lag tahmini kaldırıldığı için exponential şişme sorunu engellenir.
- Tek modele bağımlı kalmamak için validation performansına göre en iyi modellerden ağırlıklı ensemble oluşturulur.
- Guardrail, USD bazında yıldan yıla anormal +%100 / -%80 gibi sıçramaları engeller.
""")



Yorum:
- Bu sürümde 2025-2030 için USD/TRY tahmini yapılmaz; hazır kur dosyasından alınır.
- Sağlık bütçesi önce USD bazında tahmin edilir.
- TL bütçe daha sonra USD tahmini × hazır USD/TRY kuru ile türetilir.
- Recursive lag tahmini kaldırıldığı için exponential şişme sorunu engellenir.
- Tek modele bağımlı kalmamak için validation performansına göre en iyi modellerden ağırlıklı ensemble oluşturulur.
- Guardrail, USD bazında yıldan yıla anormal +%100 / -%80 gibi sıçramaları engeller.

